# Step 1 – Burn-in, pre-screening and Bayesian optimisation

Builds the herds, pre-screens the parameter space by rejection ABC, then runs Bayesian optimisation (Gaussian process, expected improvement) for every farm and loss metric.

All settings are in `config.py`. After editing it, restart the kernel.

**Outputs** (`results/optimization/`):

| File | Content |
| --- | --- |
| `observed_cases.csv` | Observed daily cases (gaps filled as set by `MISSING_DAYS_FILL`) |
| `prescreen_draws.csv` | Every prior draw with both losses |
| `prescreen_accepted.csv` | Top `PRESCREEN_ACCEPT_FRACTION` of draws per farm and metric |
| `search_bounds.csv` | Prior bounds and the narrowed bounds used by the optimiser |
| `bo_history.csv` | Every optimiser evaluation (Fig. 5A, 5B) |
| `reevaluation.csv` | Best candidates re-evaluated with `OPT_REEVAL_REPS` simulations |
| `optimal_parameters.csv` | Chosen optimum per farm, metric and run (Fig. 5C) |
| `optimal_parameters_summary.csv` | Across runs (informative when `N_OPT_RUNS` > 1) |

Burn-in herds and their summaries are in `results/burnin/`.

In [1]:
import time
import warnings

import numpy as np
import pandas as pd

import ab_sim as m
import common as c
import config as cfg
import itertools

from skopt import gp_minimize
from skopt.space import Integer, Real

from workers import prescreen_task
warnings.filterwarnings("ignore", message="The objective has been evaluated at point")

## Functions

In [2]:
def narrowed_bounds(accepted: pd.DataFrame) -> dict:
    """Search bounds for the optimiser: the range of the accepted pre-screen draws,
    widened slightly when a parameter collapsed to a single value."""
    out = {}
    for p, (lo, hi) in cfg.BOUNDS.items():
        a, b = accepted[p].min(), accepted[p].max()
        if p in cfg.INTEGER_PARAMS:
            a, b = int(a), int(b)
            if a == b:
                a, b = max(int(lo), a - 1), min(int(hi), b + 1)
        elif b - a < 1e-9:
            a, b = max(lo, a * 0.9), min(hi, b * 1.1)
        out[p] = (a, b)
    return out

In [3]:
def dimensions(bounds: dict) -> list:
    """skopt search space: integers for I and IC, log scale for T."""
    dims = []
    for p in c.PARAMS:
        lo, hi = bounds[p]
        if p in cfg.INTEGER_PARAMS:
            dims.append(Integer(int(lo), int(hi), name=p))
        else:
            prior = "log-uniform" if p in cfg.LOG_SCALE_PARAMS else "uniform"
            dims.append(Real(float(lo), float(hi), prior=prior, name=p))
    return dims

In [4]:
def mean_loss(farm: str, metric: str, params: dict, n_reps: int, seed) -> float:
    """Loss of the mean of n_reps simulated case curves against the observed series."""
    f = c.FARMS[farm]
    curves = m.simulate_case_curves(c.BURN[farm], params, f["ndays"], n_reps, seed, cfg)
    return m.loss(metric, curves.mean(axis=0), f["observed"], cfg)

In [5]:
def optimise(farm: str, metric: str, run: int, bounds: dict, accepted: pd.DataFrame):
    """One Bayesian optimisation for one farm and metric, seeded with the best pre-screen
    draws. The best distinct candidates are then re-evaluated with more simulations
    (common random numbers) and the lowest becomes the optimum.
    Returns (history, re-evaluation table, optimum)."""
    counter = itertools.count()

    def objective(x):
        params = dict(zip(c.PARAMS, x))
        params["I"], params["IC"] = int(params["I"]), int(params["IC"])
        seed = m.task_seed("bo", farm, metric, run, next(counter))
        return mean_loss(farm, metric, params, cfg.N_REPS_PER_EVAL, seed)

    seeds = accepted.nsmallest(cfg.N_SEED_POINTS_FROM_PRESCREEN, f"loss_{metric}")
    x0 = [[float(r.T), float(r.R), int(r.I), int(r.IC)] for r in seeds.itertuples()]
    y0 = [float(v) for v in seeds[f"loss_{metric}"]]
    res = gp_minimize(objective, dimensions(bounds), n_calls=cfg.N_ITERATIONS, x0=x0, y0=y0,
                      n_initial_points=cfg.N_RANDOM_START_POINTS, acq_func="EI",
                      random_state=cfg.OPT_SEED + run, noise="gaussian")

    hist = pd.DataFrame(res.x_iters, columns=c.PARAMS)
    hist["loss"] = res.func_vals
    hist["source"] = ["prescreen_seed"] * len(x0) + ["optimiser"] * (len(hist) - len(x0))
    hist["evaluation"] = np.arange(1, len(hist) + 1)
    hist["iteration"] = hist["evaluation"] - len(x0)          # <= 0 for pre-screen seeds
    hist["best_so_far"] = hist["loss"].cummin()
    first_best = hist["best_so_far"].iloc[len(x0) - 1] if len(x0) else hist["loss"].iloc[0]
    hist["relative_decrease_from_start"] = 1 - hist["best_so_far"] / first_best
    hist.insert(0, "run", run)
    hist.insert(0, "metric", metric)
    hist.insert(0, "farm", farm)

    f = c.FARMS[farm]
    cand = hist.drop_duplicates(subset=c.PARAMS).nsmallest(cfg.OPT_REEVAL_TOP, "loss")
    rows = []
    for k, r in enumerate(cand.itertuples()):
        params = {"T": float(r.T), "R": float(r.R), "I": int(r.I), "IC": int(r.IC)}
        curves = m.simulate_case_curves(c.BURN[farm], params, f["ndays"], cfg.OPT_REEVAL_REPS,
                                        m.task_seed("reeval", farm, metric, run), cfg)
        per_rep = [m.loss(metric, cv, f["observed"], cfg) for cv in curves]
        rows.append({"farm": farm, "metric": metric, "run": run, "candidate": k, **params,
                     "loss_optimiser": r.loss,
                     "loss_reeval_mean_curve": m.loss(metric, curves.mean(0), f["observed"], cfg),
                     "loss_reeval_per_sim_mean": float(np.mean(per_rep)),
                     "loss_reeval_per_sim_sd": float(np.std(per_rep, ddof=1))})
    reeval = pd.DataFrame(rows)
    best = reeval.loc[reeval["loss_reeval_mean_curve"].idxmin()]
    optimum = {"farm": farm, "metric": metric, "run": run,
               **{p: best[p] for p in c.PARAMS},
               "loss": best["loss_reeval_mean_curve"], "loss_optimiser": best["loss_optimiser"],
               "infectious_period_days": 1 / best["R"],
               "duration_of_infection_days": best["I"] + 1 / best["R"]}
    return hist, reeval, optimum

## Data and burn-in herds

In [6]:
c.ensure_dirs()
OUT = c.DIRS["optimization"]

farms = m.load_inputs(cfg)
burns = c.farm_burn_ins(farms)
c.write_burnin_summary(burns, farms)
c.FARMS, c.BURN = farms, burns          # used by optimise() in this process

pd.read_csv(c.DIRS["burnin"] / "burnin_summary_farms.csv")

Burn-in Farm1 (target 800 lactating cows)
Burn-in Farm2 (target 500 lactating cows)
Burn-in Farm3 (target 2500 lactating cows)
Burn-in Farm4 (target 3000 lactating cows)


,herd,target_lactating,lactating_at_end,total_at_end,young_stock_target,pen1_Pre-weaning,pen2_Post-weaned,pen3_Breeding,pen4_Pregnant,pen5_Springer,pen6_Fresh primiparous,pen7_Fresh multiparous,pen8_High-milk primiparous,pen9_Low-milk primiparous,pen10_High-milk multiparous,pen11_Low-milk multiparous,pen12_Dry,pen13_Close-up,pen14_Calving,pen15_Hospital (post-calving)
0,Farm1,800,800,1842,1043,117,582,86,235,22,15,74,124,63,167,248,90,12,4,3
1,Farm2,500,500,1152,652,73,364,54,147,14,11,47,82,52,81,152,63,8,1,3
2,Farm3,2500,2500,2670,1511,170,869,129,342,27,18,108,181,107,218,370,105,14,6,6
3,Farm4,3000,3000,6727,3806,428,2124,315,860,82,47,273,488,262,592,907,269,46,17,17


In [7]:
obs = pd.concat([pd.DataFrame({"farm": k, "day": np.arange(1, f["ndays"] + 1),
                               "date": f["dates"], "observed": f["observed"]})
                 for k, f in farms.items()])
obs.to_csv(OUT / "observed_cases.csv", index=False)
pd.DataFrame([{"farm": k, "days": f["ndays"], "days_filled": f["n_missing_days_filled"],
               "total_cases": f["observed"].sum(), "target_lactating": f["target_lactating"]}
              for k, f in farms.items()])

,farm,days,days_filled,total_cases,target_lactating
0,Farm1,33,14,135.0,800
1,Farm2,55,33,70.0,500
2,Farm3,23,0,605.0,2500
3,Farm4,18,3,269.0,3000


## Pre-screen (rejection ABC)

In [8]:
t0 = time.time()
rng = np.random.default_rng(cfg.OPT_SEED)
tasks = []
for farm in farms:
    prior_draws = c.sample_prior(cfg.N_PRESCREEN, rng)
    tasks += [(farm, i, c.as_params(r)) for i, r in prior_draws.iterrows()]
print(f"Pre-screen: {len(tasks)} draws x {cfg.N_REPS_PER_EVAL} simulations")

draws = pd.DataFrame(c.pool_map(prescreen_task, tasks, chunksize=16))
draws.to_csv(OUT / "prescreen_draws.csv", index=False)
print(f"done in {time.time() - t0:.0f} s")

Pre-screen: 4000 draws x 10 simulations
done in 41 s


In [9]:
k = max(1, int(round(cfg.PRESCREEN_ACCEPT_FRACTION * cfg.N_PRESCREEN)))
accepted_all, bounds_rows = [], []
accepted, bounds = {}, {}
for farm in farms:
    for metric in cfg.METRICS:
        a = draws[draws.farm == farm].nsmallest(k, f"loss_{metric}").copy()
        a.insert(1, "metric", metric)
        a["rank"] = np.arange(1, len(a) + 1)
        accepted[(farm, metric)] = a
        accepted_all.append(a)
        bounds[(farm, metric)] = narrowed_bounds(a)
        for p in c.PARAMS:
            bounds_rows.append({"farm": farm, "metric": metric, "parameter": p,
                                "prior_low": cfg.BOUNDS[p][0], "prior_high": cfg.BOUNDS[p][1],
                                "search_low": bounds[(farm, metric)][p][0],
                                "search_high": bounds[(farm, metric)][p][1]})
pd.concat(accepted_all).to_csv(OUT / "prescreen_accepted.csv", index=False)
search_bounds = pd.DataFrame(bounds_rows)
search_bounds.to_csv(OUT / "search_bounds.csv", index=False)
search_bounds

,farm,metric,parameter,prior_low,prior_high,search_low,search_high
0,Farm1,DM,T,0.0005,0.005,0.000717,0.004461
1,Farm1,DM,R,0.0400,0.080,0.042254,0.079745
2,Farm1,DM,I,1.0000,4.000,1.000000,4.000000
3,Farm1,DM,IC,1.0000,10.000,1.000000,10.000000
4,Farm1,PLF,T,0.0005,0.005,0.000670,0.004711
5,Farm1,PLF,R,0.0400,0.080,0.040090,0.079745
6,Farm1,PLF,I,1.0000,4.000,1.000000,4.000000
7,Farm1,PLF,IC,1.0000,10.000,1.000000,10.000000
8,Farm2,DM,T,0.0005,0.005,0.000755,0.004387
9,Farm2,DM,R,0.0400,0.080,0.040343,0.079428


## Bayesian optimisation

Files are rewritten after every farm and metric, so partial results survive an interruption.

In [10]:
hists, reevals, optima = [], [], []
for farm in farms:
    for metric in cfg.METRICS:
        for run in range(cfg.N_OPT_RUNS):
            t0 = time.time()
            h, r, o = optimise(farm, metric, run, bounds[(farm, metric)], accepted[(farm, metric)])
            hists.append(h)
            reevals.append(r)
            optima.append(o)
            print(f"{farm} {metric} run {run}: T={o['T']:.5f} R={o['R']:.4f} I={o['I']} "
                  f"IC={o['IC']} loss={o['loss']:.3f} ({time.time() - t0:.0f} s)")
            pd.concat(hists).to_csv(OUT / "bo_history.csv", index=False)
            pd.concat(reevals).to_csv(OUT / "reevaluation.csv", index=False)
            pd.DataFrame(optima).to_csv(OUT / "optimal_parameters.csv", index=False)

Farm1 DM run 0: T=0.00157 R=0.0667 I=1 IC=1 loss=213.182 (30 s)
Farm1 PLF run 0: T=0.00157 R=0.0667 I=1 IC=1 loss=286.766 (31 s)
Farm2 DM run 0: T=0.00221 R=0.0794 I=1 IC=1 loss=83.266 (33 s)
Farm2 PLF run 0: T=0.00098 R=0.0782 I=1 IC=8 loss=181.342 (34 s)
Farm3 DM run 0: T=0.00264 R=0.0527 I=1 IC=5 loss=204.497 (30 s)
Farm3 PLF run 0: T=0.00376 R=0.0780 I=1 IC=3 loss=415.184 (30 s)
Farm4 DM run 0: T=0.00058 R=0.0798 I=2 IC=8 loss=352.021 (33 s)
Farm4 PLF run 0: T=0.00058 R=0.0798 I=2 IC=8 loss=237.882 (36 s)


In [11]:
opt = pd.DataFrame(optima)
c.summarise(opt, ["farm", "metric"], c.PARAMS + ["loss", "infectious_period_days",
                                                "duration_of_infection_days"]
            ).to_csv(OUT / "optimal_parameters_summary.csv", index=False)
opt

,farm,metric,run,T,R,I,IC,loss,loss_optimiser,infectious_period_days,duration_of_infection_days
0,Farm1,DM,0,0.001571,0.066749,1,1,213.181729,212.832264,14.981559,15.981559
1,Farm1,PLF,0,0.001571,0.066749,1,1,286.766172,222.665880,14.981559,15.981559
2,Farm2,DM,0,0.002210,0.079428,1,1,83.266364,74.070510,12.590053,13.590053
3,Farm2,PLF,0,0.000978,0.078151,1,8,181.341928,187.710372,12.795788,13.795788
4,Farm3,DM,0,0.002640,0.052711,1,5,204.497431,252.713439,18.971355,19.971355
5,Farm3,PLF,0,0.003757,0.077983,1,3,415.184024,379.797631,12.823373,13.823373
6,Farm4,DM,0,0.000577,0.079753,2,8,352.021465,319.734848,12.538764,14.538764
7,Farm4,PLF,0,0.000577,0.079753,2,8,237.881713,232.548211,12.538764,14.538764


In [12]:
# Near-optimal parameter ranges from the Bayesian optimiser:
# evaluations whose loss is within 2 x the noise of one evaluation of the best loss
hist = pd.read_csv(OUT / "bo_history.csv")
hist = hist[hist.source == "optimiser"].copy()
hist["infectious_period_days"] = 1 / hist["R"]
hist["duration_of_infection_days"] = hist["I"] + 1 / hist["R"]
keys = ["farm", "metric", "run"]

# noise of one evaluation = SD of single-simulation losses / sqrt(simulations per evaluation)
reev = pd.read_csv(OUT / "reevaluation.csv")
noise = (reev.groupby(keys)["loss_reeval_per_sim_sd"].median()
         / np.sqrt(cfg.N_REPS_PER_EVAL)).rename("noise_sd").reset_index()
hist = hist.merge(noise, on=keys, how="left")
hist["best_loss"] = hist.groupby(keys)["loss"].transform("min")
near = hist[hist["loss"] <= hist["best_loss"] + 2 * hist["noise_sd"]]

ci_vars = c.PARAMS + ["infectious_period_days", "duration_of_infection_days"]
ci = (near.melt(id_vars=["farm", "metric"], value_vars=ci_vars,
                var_name="parameter", value_name="value")
          .groupby(["farm", "metric", "parameter"])["value"]
          .agg(n="count", median="median", mean="mean",
               p2_5=lambda x: x.quantile(0.025), p97_5=lambda x: x.quantile(0.975),
               min="min", max="max")
          .reset_index())

# Bayesian-optimisation optimum for comparison
opt = pd.read_csv(OUT / "optimal_parameters.csv")
best = opt.loc[opt.groupby(["farm", "metric"])["loss"].idxmin()].copy()
best["infectious_period_days"] = 1 / best["R"]
best["duration_of_infection_days"] = best["I"] + 1 / best["R"]
best_long = best.melt(id_vars=["farm", "metric"], value_vars=ci_vars,
                      var_name="parameter", value_name="bo_optimum")
ci = ci.merge(best_long, on=["farm", "metric", "parameter"], how="left")
ci["bo_optimum_inside_p2_5_p97_5"] = ci["bo_optimum"].between(ci["p2_5"], ci["p97_5"])

ci.to_csv(OUT / "near_optimal_intervals.csv", index=False)
ci

,farm,metric,parameter,n,median,mean,p2_5,p97_5,min,max,bo_optimum,bo_optimum_inside_p2_5_p97_5
0,Farm1,DM,I,23,2.000000,2.086957,1.000000,4.000000,1.000000,4.000000,1.000000,True
1,Farm1,DM,IC,23,3.000000,4.782609,1.000000,10.000000,1.000000,10.000000,1.000000,True
2,Farm1,DM,R,23,0.062143,0.061846,0.042254,0.079745,0.042254,0.079745,0.066749,True
3,Farm1,DM,T,23,0.001356,0.001658,0.000717,0.003674,0.000717,0.004118,0.001571,True
4,Farm1,DM,duration_of_infection_days,23,19.091851,19.272529,13.539944,25.116557,13.539944,25.666557,15.981559,True
5,Farm1,DM,infectious_period_days,23,16.091851,17.185572,12.539944,23.666557,12.539944,23.666557,14.981559,True
6,Farm1,PLF,I,91,1.000000,1.945055,1.000000,4.000000,1.000000,4.000000,1.000000,True
7,Farm1,PLF,IC,91,4.000000,4.802198,1.000000,10.000000,1.000000,10.000000,1.000000,True
8,Farm1,PLF,R,91,0.061997,0.061651,0.040090,0.079745,0.040090,0.079745,0.066749,True
9,Farm1,PLF,T,91,0.001466,0.001776,0.000670,0.004711,0.000670,0.004711,0.001571,True


In [13]:
hist = pd.read_csv(OUT / "bo_history.csv")
reev = pd.read_csv(OUT / "reevaluation.csv")
rows = []
for (farm, metric, run), h in hist.groupby(["farm", "metric", "run"]):
    h = h.sort_values("evaluation")
    seeds = h[h.source == "prescreen_seed"]
    improved = h[h["best_so_far"].diff() < 0]
    last = improved.iloc[-1] if len(improved) else h.iloc[0]
    r = reev[(reev.farm == farm) & (reev.metric == metric) & (reev.run == run)]
    noise = r["loss_reeval_per_sim_sd"].median() / np.sqrt(cfg.N_REPS_PER_EVAL)
    best = h["loss"].min()
    rows.append({
        "farm": farm, "metric": metric, "run": run,
        "best_seed_loss": seeds["loss"].min(),
        "final_best_loss": best,
        "improvement_over_seeds": seeds["loss"].min() - best,
        "noise_sd_of_one_evaluation": noise,
        "last_improvement_iteration": int(last["iteration"]),
        "share_within_2sd_of_best": float((h["loss"] <= best + 2 * noise).mean()),
        "reeval_winner_is_late_point": bool(
            len(improved) and np.isclose(
                r.loc[r["loss_reeval_mean_curve"].idxmin(), ["T", "R"]].values.astype(float),
                last[["T", "R"]].values.astype(float)).all()),
        "zero_curve_loss": m.loss(metric, np.zeros(c.FARMS[farm]["ndays"]),
                                  c.FARMS[farm]["observed"], cfg),
    })
diag = pd.DataFrame(rows)
diag

,farm,metric,run,best_seed_loss,final_best_loss,improvement_over_seeds,noise_sd_of_one_evaluation,last_improvement_iteration,share_within_2sd_of_best,reeval_winner_is_late_point,zero_curve_loss
0,Farm1,DM,0,190.756150,181.539929,9.216221,50.566340,28,0.300000,False,428.278075
1,Farm1,PLF,0,193.592493,167.742539,25.849953,389.868656,92,0.918182,False,19686.858200
2,Farm2,DM,0,79.143316,74.070510,5.072806,23.175742,50,0.409091,True,216.649351
3,Farm2,PLF,0,180.605440,180.605440,0.000000,101.943155,-9,0.463636,False,9908.926241
4,Farm3,DM,0,224.100000,224.100000,0.000000,80.581744,-9,0.263636,False,1980.521739
5,Farm3,PLF,0,383.071689,371.595226,11.476463,143.153933,36,0.918182,False,93190.594837
6,Farm4,DM,0,274.761111,274.761111,0.000000,54.596919,-9,0.190909,False,899.277778
7,Farm4,PLF,0,157.272058,157.272058,0.000000,131.949883,-9,0.572727,False,40253.304799
